# Computed fields

Às vezes queremos que essas propriedades sejam campos de verdade do modelo — ou seja, serializados como qualquer outro campo.

Para isso, o Pydantic oferece o decorador `@computed_field`.

In [1]:
from functools import cached_property
from math import pi
from pydantic import BaseModel, computed_field, Field, PydanticUserError, ValidationError

Voltando ao exemplo do `Circle`:

In [2]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @property
    def area(self):
        print("calculating area...")
        return pi * self.radius ** 2

Para transformar a propriedade num campo do modelo (um campo **calculado**), basta decorá-la. Porém, como agora é um campo de um modelo
Pydantic, **PRECISAMOS** anotar o tipo de retorno da propriedade, para o Pydantic saber o tipo do campo.

Se esquecer, recebemos uma `PydanticUserError` já na definição da classe:

In [3]:
try:
    class Circle(BaseModel):
        center: tuple[int, int] = (0, 0)
        radius: int = Field(default=1, gt=0, frozen=True)

        @computed_field
        @property
        def area(self):
            print("calculating area...")
            return pi * self.radius ** 2
except PydanticUserError as ex:
    print(f"{type(ex).__name__}: {ex}")

PydanticUserError: Computed field is missing return type annotation or specifying `return_type` to the `@computed_field` decorator (e.g. `@computed_field(return_type=int | str)`)

For further information visit https://errors.pydantic.dev/2.13/u/model-field-missing-annotation


A exceção diz que podemos informar o tipo no *type hint* ou como argumento do decorador — o mais comum é o *type hint*:

In [4]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @computed_field
    @property
    def area(self) -> float:
        print("calculating area...")
        return pi * self.radius ** 2

Agora, criando uma instância:

In [5]:
c = Circle()

In [6]:
print(f"c: {c!r}")

calculating area...
c: Circle(center=(0, 0), radius=1, area=3.141592653589793)


`area` aparece na representação e nos dados serializados:

In [7]:
print(f"c.model_dump(): {c.model_dump()!r}")

calculating area...
c.model_dump(): {'center': (0, 0), 'radius': 1, 'area': 3.141592653589793}


In [8]:
print(f"c.model_dump_json(): {c.model_dump_json()!r}")

calculating area...
c.model_dump_json(): '{"center":[0,0],"radius":1,"area":3.141592653589793}'


Podemos dar um alias ao campo calculado (talvez não aqui, mas em geral dá):

In [9]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @computed_field(alias="AREA")
    @property
    def area(self) -> float:
        print("calculating area...")
        return pi * self.radius ** 2

In [10]:
c = Circle()
print(f"c.model_dump(by_alias=True): {c.model_dump(by_alias=True)!r}")

calculating area...
c.model_dump(by_alias=True): {'center': (0, 0), 'radius': 1, 'AREA': 3.141592653589793}


Também dá para omitir campos da representação da instância:

In [11]:
class Model(BaseModel):
    field_1: int = Field(repr=False)
    field_2: int

In [12]:
m = Model(field_1=1, field_2=2)
print(f"m: {m!r}")

m: Model(field_2=2)


`field_1` não aparece na representação, mas continua sendo um campo normal: acessível com ponto e serializado normalmente:

In [13]:
m.field_1=10
print(f"m.field_1: {m.field_1!r}")

m.field_1: 10


In [14]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'field_1': 10, 'field_2': 2}


Nas propriedades calculadas não temos o `Field` para omitir da representação, mas o decorador `@computed_field` oferece a mesma opção,
também com o argumento `repr`:

In [15]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @computed_field(alias="AREA", repr=False)
    @property
    def area(self) -> float:
        print("calculating area...")
        return pi * self.radius ** 2

In [16]:
c = Circle()

In [17]:
print(f"c: {c!r}")

c: Circle(center=(0, 0), radius=1)


`area` não aparece na representação, mas é acessível normalmente:

In [18]:
print(f"c.area: {c.area!r}")

calculating area...
c.area: 3.141592653589793


e serializa normalmente:

In [19]:
print(f"c.model_dump(): {c.model_dump()!r}")

calculating area...
c.model_dump(): {'center': (0, 0), 'radius': 1, 'area': 3.141592653589793}


Um campo calculado é **somente leitura**:

In [20]:
try:
    c.area = 10
except AttributeError as ex:
    print(f"{type(ex).__name__}: {ex}")

AttributeError: property 'area' of 'Circle' object has no setter


O Pydantic até suporta *setter* para campo calculado — mas o autor não vê caso de uso realista, então não é coberto aqui.

É simples e, se precisar, está documentado [aqui](https://docs.pydantic.dev/latest/concepts/fields/#the-computed_field-decorator).

Campos calculados também funcionam com propriedades cacheadas:

In [21]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @computed_field(alias="AREA", repr=False)
    @cached_property
    def area(self) -> float:
        print("calculating area...")
        return pi * self.radius ** 2

In [22]:
c = Circle()
print(f"c.area: {c.area!r}")

calculating area...
c.area: 3.141592653589793


In [23]:
print(f"c.area: {c.area!r}")

c.area: 3.141592653589793


O cache funcionou: a segunda chamada não recalculou a área.

> 💡 **Complemento:** campos calculados aparecem no JSON Schema de **serialização** (o que o modelo emite), não no de validação (o que
> ele aceita) — faz sentido, já que ninguém envia `area` na entrada:

In [24]:
# o JSON Schema usa o alias ("AREA") como chave da propriedade
serialization_props = Circle.model_json_schema(mode="serialization")["properties"]
validation_props = Circle.model_json_schema(mode="validation")["properties"]
print(f"schema de serialização: {sorted(serialization_props)}")
print(f"schema de validação: {sorted(validation_props)}")

schema de serialização: ['AREA', 'center', 'radius']
schema de validação: ['center', 'radius']
